---
image: example.gif
pub-info:
    abstract: |
        Compares several ways of visualising queues that grow too large to show as individual icons,
        going beyond the default '+ x more' text. A deep dive for anyone whose model produces queues
        large enough to make the standard display hard to read at a glance.
execute: 
  enabled: true
---

# Deep Dive: Ways of visualising larger queues

While vidigi originally provided only a simple way to see the size of larger queues (via the '+ x more' text that appears when the snapshot max is exceeded), this is not particularly visually intuitive for understanding the magnitude of the different queues that are building up. 

We could use the synchronised subplot approach from the [Additional Synchronised Traces - Orthopaedic Ward - Hospital Efficiency Project](https://hsma-tools.github.io/vidigi/examples/example_13_additional_synchronised_traces_method_1/synchronised_traces.html) example or the [gas station](https://hsma-tools.github.io/vidigi/examples/example_15_gas_station_refuelling/gas_station.html) example, but when we have multiple steps it could still be challenging to quickly get a sense of where the queues are as your users will have to look across at different parts of the page, and it can be quite a cumbersome thing to add in.  

Instead, it would be better to be able to quickly get a sense of queue size directly next to the relevant step. This poses some challenges in vidigi and there are some limitations - but it is possible! 

Some of what is shown in this example has now been incorporated into vidigi as an official alternative to the '+ x more' snapshot as part of release 1.1.0, and the '+ x more' text can be swapped out for gauges simply by setting `step_snapshot_limit_gauges=True` in either `generate_animation_df()` or `animate_activity_log()`. There is also a helper function - prep.ascii_queue_icon() - that could be used to add gauges in manually between the `generate_animation_df()` and `generate_animation()` steps. 

However, seeing under the hood will hopefully help you adapt this approach for your own more complex animations.

---

We'll be working with some of the outputs from the first 'resourceless' animation example - the [mental health appointment booking](https://hsma-tools.github.io/vidigi/examples/example_11_resourceless_animation/ex_11_resourceless.html) model. 

By the end of that model, we have some queues with as few as 28 people waiting, and some with as high as 462! But because of the way it's displayed, it's not easy to get a sense of that. 

![](assets/2025-08-07-16-05-09.png)

Let's kick off with our imports. We'll be using the separate functions so we can make some changes to the dataframes that get produced along the way. 

In [ ]:
import pandas as pd
import numpy as np
import re
from vidigi.prep import reshape_for_animations, generate_animation_df
from vidigi.animation import generate_animation
import plotly.io as pio
pio.renderers.default = "notebook"
WARM_UP = 60 * 1
RESULTS_COLLECTION = 90 * 1

We've saved the event log produced by the mental health model, so let's import it here. 

In [ ]:
event_log_df = pd.read_csv("as-is.csv")

We kick off with a pretty normal call of reshape_for_animations, other than setting the step_snapshot_max to be far higher than what is usually seen, and ensuring it's so high that the model queue will never reach this level (ensuring we get every entity at every stage). 

In [ ]:
full_patient_df = reshape_for_animations(
        event_log_df,
        entity_col_name="patient",
        limit_duration=WARM_UP+RESULTS_COLLECTION,
        every_x_time_units=1,
        step_snapshot_max=9999999,
        )

# Remove the warm-up period from the event log
full_patient_df = full_patient_df[full_patient_df["snapshot_time"] >= WARM_UP]

Now, for only the stage we want to display as bars instead of individual entities, we will count the total number of patients queueuing at that stage.

We also need to give each event within this new dataframe a consistent entity ID across all snapshot_times so it will animate correctly. 

In [ ]:
reshaped_df = full_patient_df.groupby(['snapshot_time', 'event', 'event_type'])['patient'].count().reset_index()

reshaped_df = reshaped_df[(reshaped_df["event"].str.contains("appointment_booked_waiting")) | reshaped_df["event"].str.contains("referred_out")]

event_id_map = {event: idx for idx, event in enumerate(reshaped_df['event'].unique())}

reshaped_df['entity_id'] = reshaped_df['event'].map(event_id_map)
reshaped_df['entity_id'] = reshaped_df['entity_id'].apply(lambda x: f"BAR{x}")
reshaped_df['time'] = reshaped_df['snapshot_time']

reshaped_df = reshaped_df.rename(columns={'patient': 'patient_count'})

reshaped_df.head(40)

OPTIONAL - but recommend. 

Let's ensure that there's a count of patients referred out at every point in time. This will ensure we don't end up with the counts appearing and disappearing across the course of the animation. 

In [ ]:
# 1a. Extract unique snapshot_times
snapshot_times = reshaped_df['snapshot_time'].unique()

# 1b. Extract unique 'referred_out' event values only
referred_out_events = reshaped_df.loc[reshaped_df['event'].str.startswith('referred_out'), 'event'].unique()

# 2. Create a MultiIndex of all combinations of snapshot_time and referred_out_events
full_index = pd.MultiIndex.from_product(
    [snapshot_times, referred_out_events],
    names=['snapshot_time', 'event']
)

# 3. Set index of the df to ['snapshot_time', 'event'] to facilitate reindexing
df_referred = reshaped_df.set_index(['snapshot_time', 'event'])

# 4. Reindex to full index, filling missing rows with patient_count = 0
df_referred = df_referred.reindex(full_index).reset_index()

# 5. For the new rows where patient_count was filled, fill missing columns like event_type, entity_id, time
df_referred['patient_count'] = df_referred['patient_count'].fillna(0)
df_referred['event_type'] ='queue'
df_referred['time'] = df_referred['snapshot_time']

# Build a dictionary mapping each referred_out event to its corresponding entity_id
event_to_entity = reshaped_df.loc[
    reshaped_df['event'].str.startswith('referred_out'),
    ['event', 'entity_id']
].drop_duplicates().set_index('event')['entity_id'].to_dict()

# Assign using map
df_referred['entity_id'] = df_referred['event'].map(event_to_entity)

# 6. Join this back into our dataframe
final_count_df = pd.concat([
    reshaped_df[reshaped_df['event'].str.contains("appointment_booked_waiting")],
    df_referred
])

We then join this back to the original dataframe, ensuring we first get rid of the entity-level rows for the stage we've just done the counts for. 

In [ ]:
full_patient_df = pd.concat(
    [full_patient_df[(~full_patient_df["event"].str.contains("appointment_booked_waiting")) &
                     (~full_patient_df["event"].str.contains("referred_out"))].rename(columns={"patient": "entity_id"}),
    final_count_df],
    ignore_index=True
)

# If we don't fill the NA values with a float number, we encounter issues later where certain actions can't be performed due to the data type of missing values.
full_patient_df["patient_count"] = full_patient_df["patient_count"].fillna(0.0)

full_patient_df = full_patient_df.sort_values(["snapshot_time", "time", "entity_id"])


Now let's read in our entity positioning dataframe and carry on with the animation steps. 

In [ ]:
event_position_df = pd.read_csv("as-is_event_position_df.csv")

# We just use this line to slightly modify the position of one set of our events
event_position_df['x'] = event_position_df.apply(
    lambda row: row['x'] - 120
    if "appointment_booked_waiting" in row['event']
    else row['x'],  # fallback to original value
    axis=1
)

event_position_df

In [ ]:
full_df_plus_pos = generate_animation_df(
                full_entity_df=full_patient_df,
                entity_col_name="entity_id",
                event_position_df=event_position_df,
                wrap_queues_at=25,
                step_snapshot_max=9999999,
                gap_between_entities=20,
                gap_between_queue_rows=15,
                debug_mode=True
        )

We're now going to define a new function that helps us to swap out the counts for an interpretable bar. 

In [ ]:
def ascii_queue_icon(count, max_count, bar_length=10, filled_char="█", empty_char="░", count_only=False):
    """
    Returns an ASCII progress bar string representing the queue length.

    Args:
        count (int): The current patient count.
        max_count (int): The maximum patient count in the data.
        bar_length (int): Total length of the bar in characters.
        filled_char (str): Character to use for filled segments.
        empty_char (str): Character to use for empty segments.

    Returns:
        str: ASCII progress bar string.
    """
    if count_only:
        if isinstance(count, str):
            return count
        else:
            return f"{count:.0f}"
    else:
        if max_count == 0:
            return empty_char * bar_length  # avoid division by zero

        if not np.isnan(count):
            filled_len = int(round(bar_length * count / max_count))
            bar = filled_char * filled_len + empty_char * (bar_length - filled_len)
            return f"[{bar}] {count:.0f}"


Let's now add this into our dataframe. 

In [ ]:
full_df_plus_pos['patient_cumulative'] = full_df_plus_pos.sort_values('snapshot_time').groupby(['event'])['patient_count'].cumsum()
full_df_plus_pos['patient_cumulative_count_display'] = full_df_plus_pos.apply(
    lambda row: f"{row['patient_cumulative']:.0f} (+{row['patient_count']:.0f})",  # fallback to original value
    axis=1
)

In [ ]:
# Calculate max patient count only from relevant events
max_count = full_df_plus_pos.loc[
    full_df_plus_pos['event'].str.contains("appointment_booked_waiting"), 'patient_count'
].max()

# Update the icon column conditionally
full_df_plus_pos['icon'] = full_df_plus_pos.apply(
    lambda row: ascii_queue_icon(row['patient_count'], max_count, 20)
    if "appointment_booked_waiting" in row['event']
    else row['icon'],  # fallback to original value
    axis=1
)

full_df_plus_pos['icon'] = full_df_plus_pos.apply(
    lambda row: ascii_queue_icon(row['patient_cumulative_count_display'], max_count, count_only=True)
    if "referred_out" in row['event']
    else row['icon'],  # fallback to original value
    axis=1
)

# Also move the icons over a bit
full_df_plus_pos['x_final'] = full_df_plus_pos.apply(
    lambda row: row['x_final'] - 100
    if "appointment_booked_waiting" in row['event']
    else row['x_final'],  # fallback to original value
    axis=1
)

full_df_plus_pos['x_final'] = full_df_plus_pos.apply(
    lambda row: row['x_final'] - 50
    if "referred" in row['event']
    else row['x_final'],  # fallback to original value
    axis=1
)

From the sample below, we can see some examples of what the icon will now look like. 

In [ ]:
full_df_plus_pos[full_df_plus_pos["event"].str.contains("appointment_booked_waiting")]

With this done, we can generate our animation!

In [ ]:
generate_animation(
        full_entity_df_plus_pos=full_df_plus_pos,
        event_position_df=event_position_df,
        scenario=None,
        plotly_height=800,
        plotly_width=1000,
        override_x_max=1000,
        override_y_max=1000,
        entity_icon_size=10,
        text_size=10,
        include_play_button=True,
        add_background_image=None,
        display_stage_labels=True,
        time_display_units="d",
        simulation_time_unit="days",
        start_date="2022-06-27",
        setup_mode=False,
        frame_duration=1500, #milliseconds
        frame_transition_duration=1000, #milliseconds
        debug_mode=False
    )

Note that the flow of the patients to their actual appointments is somewhat unintuitive. One simple option is to remove the play button option, which means users can only scrub. This removes any frame interpolation, leading to a more interpretable plot. 

In [ ]:
generate_animation(
        full_entity_df_plus_pos=full_df_plus_pos,
        event_position_df=event_position_df,
        scenario=None,
        plotly_height=800,
        plotly_width=1000,
        override_x_max=1000,
        override_y_max=1000,
        entity_icon_size=10,
        text_size=10,
        include_play_button=False,   # CHANGED
        add_background_image=None,
        display_stage_labels=True,
        time_display_units="d",
        simulation_time_unit="days",
        start_date="2022-06-27",
        setup_mode=False,
        # frame_duration=1500, #milliseconds
        # frame_transition_duration=1000, #milliseconds
        debug_mode=False
    )

We can achieve something similar while not losing the play button by turning the play button back on and **setting the frame transition duration to 0**. 

In [ ]:
generate_animation(
        full_entity_df_plus_pos=full_df_plus_pos,
        event_position_df=event_position_df,
        scenario=None,
        plotly_height=800,
        plotly_width=1000,
        override_x_max=1000,
        override_y_max=1000,
        entity_icon_size=10,
        text_size=10,
        include_play_button=True,
        add_background_image=None,
        display_stage_labels=True,
        time_display_units="d",
        simulation_time_unit="days",
        start_date="2022-06-27",
        setup_mode=False,
        frame_duration=1500, #milliseconds
        frame_transition_duration=0, #milliseconds   # CHANGED
        debug_mode=False
    )

Now, in this particular example, it perhaps doesn't feel like we're getting a huge amount more information than we would from any old animated bar chart. 

So let's add the final LOS of each individual at the point they attend the clinic, and whether they were a priority patient. 

In [ ]:
def show_priority_icon(row):
    if "have_appointment" in row["event"]:
        if int(row["pathway"]) == 2:
            return "🚨"
        else:
            return row["icon"]
    else:
        return row["icon"]

def add_los_to_icon(row):
    if "have_appointment" in row["event"]:
        return f'{row["icon"]}<br>{int(row["wait"])}'
    else:
        return row["icon"]

In [ ]:
full_df_plus_pos = full_df_plus_pos.assign(
    icon=full_df_plus_pos.apply(show_priority_icon, axis=1)
    )

full_df_plus_pos = full_df_plus_pos.assign(
    icon=full_df_plus_pos.apply(add_los_to_icon, axis=1)
    )

In [ ]:
generate_animation(
        full_entity_df_plus_pos=full_df_plus_pos,
        event_position_df=event_position_df,
        scenario=None,
        plotly_height=800,
        plotly_width=1200,
        override_x_max=1000,
        override_y_max=1000,
        entity_icon_size=10,
        text_size=10,
        include_play_button=True,
        add_background_image=None,
        display_stage_labels=True,
        time_display_units="d",
        simulation_time_unit="days",
        start_date="2022-06-27",
        setup_mode=False,
        frame_duration=1000, #milliseconds
        frame_transition_duration=0, #milliseconds   # CHANGED
        debug_mode=False
    )

Values below the individuals show the wait time in days. 

Priority patients are identified with a 🚨 symbol. These patients will be moved to the front of the booking queue. 

# Exploring experimental alternative 1: Replacing 'x more' with ascii count bar

In [ ]:
def ascii_queue_icon(icon, count, max_count, bar_length=10, filled_char="█", empty_char="░", count_only=False):
    """
    Returns an ASCII progress bar string representing the queue length.

    Args:
        icon (str): The current icon
        count (int): The current patient count.
        max_count (int): The maximum patient count in the data.
        bar_length (int): Total length of the bar in characters.
        filled_char (str): Character to use for filled segments.
        empty_char (str): Character to use for empty segments.

    Returns:
        str: ASCII progress bar string.
    """
    if max_count == 0:
        return empty_char * bar_length  # avoid division by zero

    if not np.isnan(count):
        if count_only:
            return f"{count:.0f}"
        else:
            if "more" in icon:
                filled_len = int(round(bar_length * count / max_count))
                bar = filled_char * filled_len + empty_char * (bar_length - filled_len)
                return f"[{bar}] + {count:.0f} more"
            else:
                return icon


    else:
        return ""


In [ ]:
event_log_df = pd.read_csv("as-is.csv")
full_patient_df = reshape_for_animations(
        event_log_df,
        entity_col_name="patient",
        limit_duration=WARM_UP+RESULTS_COLLECTION,
        every_x_time_units=1,
        step_snapshot_max=10,
        )

# Remove the warm-up period from the event log
full_patient_df = full_patient_df[full_patient_df["snapshot_time"] >= WARM_UP]

event_position_df = pd.read_csv("as-is_event_position_df.csv")

event_position_df['x'] = event_position_df.apply(
    lambda row: row['x'] - 70
    if "appointment_booked_waiting" in row['event']
    else row['x'],  # fallback to original value
    axis=1
)

full_df_plus_pos = generate_animation_df(
                full_entity_df=full_patient_df,
                entity_col_name="patient",
                event_position_df=event_position_df,
                wrap_queues_at=10,
                step_snapshot_max=50,
                gap_between_entities=15,
                gap_between_queue_rows=20,
                debug_mode=True
        )

full_df_plus_pos['patient_count'] = full_df_plus_pos.apply(
    lambda row: int(re.search(r'\d+', row['icon'])[0])
    if "more" in row['icon']
    else 0,
    axis=1
)

max_count = max(full_df_plus_pos['patient_count'])

In [ ]:
# Update the icon column conditionally
full_df_plus_pos['icon'] = full_df_plus_pos.apply(
    lambda row: ascii_queue_icon(row['icon'], row['patient_count'], max_count, 10)
    if "appointment_booked_waiting" in row['event']
    else row['icon'],  # fallback to original value
    axis=1
)

# Also move the icon over a bit
full_df_plus_pos['x_final'] = full_df_plus_pos.apply(
    lambda row: row['x_final'] - 100
    if "appointment_booked_waiting" in row['event']
    else row['x_final'],  # fallback to original value
    axis=1
)

In [ ]:
generate_animation(
        full_entity_df_plus_pos=full_df_plus_pos,
        event_position_df=event_position_df,
        entity_col_name="patient",
        scenario=None,
        plotly_height=1000,
        plotly_width=1000,
        override_x_max=1000,
        override_y_max=1000,
        entity_icon_size=10,
        text_size=10,
        include_play_button=True,
        add_background_image=None,
        display_stage_labels=True,
        time_display_units="d",
        simulation_time_unit="days",
        start_date="2022-06-27",
        setup_mode=False,
        frame_duration=1500, #milliseconds
        frame_transition_duration=1000, #milliseconds
        debug_mode=False
    )

Let's again try setting our frame duration to 0. 

In [ ]:
generate_animation(
        full_entity_df_plus_pos=full_df_plus_pos,
        event_position_df=event_position_df,
        entity_col_name="patient",
        scenario=None,
        plotly_height=1000,
        plotly_width=1000,
        override_x_max=1000,
        override_y_max=1000,
        entity_icon_size=10,
        text_size=10,
        include_play_button=True,
        add_background_image=None,
        display_stage_labels=True,
        time_display_units="d",
        simulation_time_unit="days",
        start_date="2022-06-27",
        setup_mode=False,
        frame_duration=1500, #milliseconds
        frame_transition_duration=30, #milliseconds
        debug_mode=False
    )